In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# qwen_vqa_5q_to_pali_jsonl.py
from __future__ import annotations
import os, sys, json, re, difflib, unicodedata
from pathlib import Path
from typing import List, Tuple, Optional
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

# ===================== USER SETTINGS =====================
IMAGES_DIR    = "/content/drive/MyDrive/Combined/Combined images"
OUT_JSONL     = "/content/drive/MyDrive/ALL IN_pg/VQA new/vqa_5q_new.jsonl"
MODE          = "resume"        # "overwrite" | "resume"
SEED          = 42

# PaLI-Gemma wants the <image> token in the prefix
PALI_PREFIX   = "<image>\n"     # keep newline
MAX_NEW_TOKS  = 8               # short so Qwen emits a single label
DEBUG_RAW     = False           # write raw model text for auditing; set False later
# Optional env overrides:
#   MODEL_ID=Qwen/Qwen2.5-VL-7B-Instruct
#   BATCH_SIZE (unused here; we run per-question to keep it robust on VRAM)
#   QUANTIZE_4BIT=1
#   USE_FAST_PROCESSOR=1
# =========================================================

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

QUESTIONS = [
    {"q": "Is there any building visible?",            "choices": ["yes","no","uncertain"]},
    {"q": "Predominant building use?",                  "choices": ["residential","industrial","commercial","mixed","unknown"]},
    {"q": "Dominant roof appearance?",                  "choices": ["metal","concrete/flat","tile","mixed","unknown"]},
    {"q": "Typical building footprint shape?",          "choices": ["rectangular","L-shaped","irregular","mixed","unknown"]},
    {"q": "Is active construction visible?",          "choices": ["yes","no","uncertain"]},
]

# A very literal system prompt to force single-label outputs
VQA_SYSTEM = (
    "You are an expert annotator of nadir-view aerial/satellite images. "
    "For each question, answer with exactly ONE label from the given list. "
    "Return ONLY the label text—no punctuation, no explanation."
)

# ---------- helpers ----------
def find_images(root: str | Path) -> List[Path]:
    root = Path(root)
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in EXTS)

def load_image_rgb(p: Path) -> Optional[Image.Image]:
    try:
        img = Image.open(p)
        if img.mode != "RGB":
            img = img.convert("RGB")
        return img
    except Exception as e:
        print(f"[WARN] Could not open {p}: {e}", file=sys.stderr)
        return None

def _norm(s: str) -> str:
    # normalize unicode, lowercase, collapse spaces
    s = unicodedata.normalize("NFKC", s).strip().lower()
    s = " ".join(s.split())
    return s

ALIASES = {
    "flat": "concrete/flat",
    "concrete": "concrete/flat",
    "concrete flat": "concrete/flat",
    "l shaped": "L-shaped",
    "l-shape": "L-shaped",
    "lshape": "L-shaped",
    "lshaped": "L-shaped",
    "l shaped footprint": "L-shaped",
    "rect": "rectangular",
    "rectangle": "rectangular",
    "rectangular footprint": "rectangular",
}

def map_to_choice(text: str, choices: List[str]) -> Optional[str]:
    """Map Qwen's output to one of the allowed labels. Returns None if not confident."""
    t = _norm(text)
    # 1) Exact / startswith match
    for c in choices:
        cn = _norm(c)
        if t == cn or t.startswith(cn):
            return c
    # 2) Alias table
    if t in ALIASES and ALIASES[t] in choices:
        return ALIASES[t]
    # 3) Fuzzy match with decent cutoff
    cand = difflib.get_close_matches(t, [_norm(c) for c in choices], n=1, cutoff=0.82)
    if cand:
        for c in choices:
            if _norm(c) == cand[0]:
                return c
    return None

def build_vqa_prompt(processor, pil_img: Image.Image, question: str, choices: List[str]) -> str:
    # strict user message, echo choices as Python list and in words
    user_text = (
        f"Answer the following strictly with one label.\n"
        f"Question: {question}\n"
        f"Allowed labels: {choices}\n"
        f"Return ONLY one of these labels (verbatim)."
    )
    messages = [
        {"role": "system", "content": VQA_SYSTEM},
        {"role": "user", "content": [
            {"type": "image", "image": pil_img},
            {"type": "text",  "text": user_text},
        ]}
    ]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def qwen_answer_label(model, processor, pil_img: Image.Image, question: str, choices: List[str],
                      max_new_tokens: int = MAX_NEW_TOKS) -> Tuple[str, str]:
    """Returns (mapped_label, raw_text). Falls back to 'unknown'/'uncertain'/last choice if needed."""
    # Pass 1: deterministic (beams)
    text = build_vqa_prompt(processor, pil_img, question, choices)
    inp = processor(text=text, images=pil_img, return_tensors="pt")
    for k, v in list(inp.items()):
        if hasattr(v, "to"):
            inp[k] = v.to(model.device)
    out = model.generate(**inp, do_sample=False, num_beams=3, max_new_tokens=max_new_tokens)
    pl = inp["input_ids"].shape[1]
    raw = processor.tokenizer.decode(out[0, pl:], skip_special_tokens=True).strip()
    mapped = map_to_choice(raw, choices)
    if mapped is not None:
        return mapped, raw

    # Pass 2: gentle sampling retry with stricter wording
    retry_text = text + "\nReturn ONLY one allowed label. If unsure, choose 'unknown' or 'uncertain' if available."
    inp2 = processor(text=retry_text, images=pil_img, return_tensors="pt")
    for k, v in list(inp2.items()):
        if hasattr(v, "to"):
            inp2[k] = v.to(model.device)
    out2 = model.generate(**inp2, do_sample=True, temperature=0.2, top_p=0.9, max_new_tokens=max_new_tokens)
    raw2 = processor.tokenizer.decode(out2[0, inp2["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    mapped2 = map_to_choice(raw2, choices)
    if mapped2 is not None:
        return mapped2, raw2

    # Final fallback to a safe "unknown"/"uncertain" if available
    if "unknown" in choices:
        return "unknown", raw2
    if "uncertain" in choices:
        return "uncertain", raw2
    return choices[-1], raw2

def pick_model_and_device() -> Tuple[str, str]:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    env_mid = os.getenv("MODEL_ID", "").strip() or "Qwen/Qwen2.5-VL-7B-Instruct"
    return env_mid, device

def load_qwen(model_id: str, device: str):
    use_fast = os.getenv("USE_FAST_PROCESSOR", "1") != "0"
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, use_fast=use_fast)
    kwargs = dict(trust_remote_code=True)
    if device == "cuda" and os.getenv("QUANTIZE_4BIT", "0") == "1":
        kwargs.update(load_in_4bit=True, device_map="auto")
    elif device == "cuda":
        kwargs.update(torch_dtype=torch.bfloat16, device_map="auto")
    else:
        kwargs.update(torch_dtype=torch.float32)
    model = AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
    model.eval()
    return processor, model

def main():
    torch.manual_seed(SEED)
    model_id, device = pick_model_and_device()

    print(f"[INFO] Device: {device}")
    if device == "cuda":
        print(f"[INFO] GPU: {torch.cuda.get_device_name(0)} "
              f"({torch.cuda.get_device_properties(0).total_memory/(1024**3):.1f} GB)")
        torch.backends.cuda.matmul.allow_tf32 = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass
    print(f"[INFO] Model: {model_id}")

    images_all = find_images(IMAGES_DIR)
    if not images_all:
        raise SystemExit(f"No images found in {IMAGES_DIR}")
    print(f"[INFO] Found {len(images_all)} images under {IMAGES_DIR}")

    processor, model = load_qwen(model_id, device)

    p_out = Path(OUT_JSONL).expanduser().resolve()
    p_out.parent.mkdir(parents=True, exist_ok=True)

    # Resume/overwrite handling — we track (image_rel, question) pairs
    seen_pairs = set()
    if MODE.lower() == "overwrite":
        if p_out.exists():
            print("[INFO] MODE=overwrite → deleting existing output")
            p_out.unlink()
        print("[INFO] Fresh run (no resume).")
    else:
        if p_out.exists():
            with open(p_out, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        obj = json.loads(line)
                        if isinstance(obj, dict) and "image" in obj and "prefix" in obj:
                            # recover question from prefix after 'question: ' and before ' answer:'
                            pref = obj["prefix"]
                            m = re.search(r"question:\s*(.*?)\s*answer:\s*$", pref, flags=re.IGNORECASE)
                            q = m.group(1).strip() if m else None
                            if q:
                                seen_pairs.add((obj["image"], q))
                    except Exception:
                        pass
            print(f"[INFO] Resuming. Found {len(seen_pairs)} (image,question) lines already written.")
        else:
            print("[INFO] Fresh run (no prior JSONL).")

    wrote = 0
    with open(p_out, "a", encoding="utf-8") as fout:
        for idx, p_abs in enumerate(images_all, 1):
            img = load_image_rgb(p_abs)
            if img is None:
                print(f"[{idx}/{len(images_all)}] SKIP (failed to load): {p_abs}")
                continue
            rel = Path(os.path.relpath(p_abs, IMAGES_DIR)).as_posix()

            for spec in QUESTIONS:
                q = spec["q"]; choices = spec["choices"]
                if (rel, q) in seen_pairs:
                    continue

                label, raw = qwen_answer_label(model, processor, img, q, choices, max_new_tokens=MAX_NEW_TOKS)

                line = {
                    "image": rel,
                    "prefix": f"{PALI_PREFIX}question: {q} answer:",
                    "suffix": label
                }
                if DEBUG_RAW:
                    line["choices"] = choices
                    line["raw"] = raw

                fout.write(json.dumps(line, ensure_ascii=False) + "\n")
                wrote += 1
                seen_pairs.add((rel, q))

            if idx % 25 == 0:
                print(f"[{idx}/{len(images_all)}] processed; total lines written so far: {wrote}")

    print(f"[INFO] Done. Wrote {wrote} new rows → {p_out}")

if __name__ == "__main__":
    main()


[INFO] Device: cuda
[INFO] GPU: NVIDIA A100-SXM4-40GB (39.6 GB)
[INFO] Model: Qwen/Qwen2.5-VL-7B-Instruct
[INFO] Found 1024 images under /content/drive/MyDrive/Combined/Combined images


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

[INFO] Fresh run (no prior JSONL).


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[25/1024] processed; total lines written so far: 125
[50/1024] processed; total lines written so far: 250
[75/1024] processed; total lines written so far: 375
[100/1024] processed; total lines written so far: 500
[125/1024] processed; total lines written so far: 625
[150/1024] processed; total lines written so far: 750
[175/1024] processed; total lines written so far: 875
[200/1024] processed; total lines written so far: 1000
[225/1024] processed; total lines written so far: 1125
[250/1024] processed; total lines written so far: 1250
[275/1024] processed; total lines written so far: 1375
[300/1024] processed; total lines written so far: 1500
[325/1024] processed; total lines written so far: 1625
[350/1024] processed; total lines written so far: 1750
[375/1024] processed; total lines written so far: 1875
[400/1024] processed; total lines written so far: 2000
[425/1024] processed; total lines written so far: 2125
[450/1024] processed; total lines written so far: 2250
[475/1024] processed